# Round 9 — Behavior-conditioned support evidence

**One independent, bounded feature experiment.** Twelve new CPU readouts, six saved controls, no encoder calls or downloads. The repeatedly inspected 881-comment development cohort is not a fresh holdout or a Kaggle score.

This round was frozen together with its companion before either result was available. Neither uses the other round’s outcomes.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/behavior_support_features.json").is_file())
from scripts.run_behavior_support_features import figures, write_dashboard, json_hash
from jigsaw_rules.runtime import digest
config = json.loads((ROOT / "configs/behavior_support_features.json").read_text())
print("Primary:", config["primary"])
print("New fits:", config["new_fits"], "| Saved controls:", config["cached_control_readouts"])


Primary: conditioned_all
New fits: 12 | Saved controls: 6


## 1. Measured starting point

Round 7’s combined pair features reached 0.720881 macro AUC, below raw basic geometry (0.723068). Its primary failed. That result is preserved, not presented as an accepted improvement. The table below is loaded from its hash-pinned report.

In [2]:
previous_path = ROOT / "reports/matched_support_features/results.json"
assert digest(previous_path) == config["round7_results_sha256"]
previous = json.loads(previous_path.read_text())
display(pd.DataFrame(previous["pooled_metrics"]))
print("Prior decision:", previous["decision"])


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,paired_global,0.720518,0.734005,0.736241
4,paired_local,0.722379,0.735450,0.736958
5,paired_all,0.720881,0.735856,0.736589
6,repaired_all,0.718273,0.733251,0.733773
7,orientation_all,0.721683,0.734279,0.735956
8,unnormalized_all,0.721144,0.735128,0.736415


Prior decision: DO_NOT_PROMOTE_PRIMARY


## 2. Registered construction and evidence integrity

Twelve existing observable action/context descriptors condition semantic evidence from same-rule references. Four contrasts per descriptor give 48 new features. Direct-descriptor, uniform-evidence and descriptor-permutation controls isolate whether conditional exemplar evidence adds value.

The terminal helper computes first and records the immutable result. This notebook verifies and displays it; opening or replaying it never silently trains models.

In [3]:
result_path = ROOT / "reports/behavior_support_features/results.json"
if not result_path.is_file():
    raise RuntimeError("Run the corresponding bounded helper first.")
result = json.loads(result_path.read_text())
marker_path = ROOT / "runs/behavior_support_features" / result["run_id"] / "finished.json"
marker = json.loads(marker_path.read_text())
assert marker["public_hashes"]["results.json"] == digest(result_path)
assert marker["identity"] == result["identity"]
assert json_hash(result["identity"])[:20] == result["run_id"]
CHARTS = figures(result)
print("Run:", result["run_id"], "| Decision:", result["decision"])
print("Control parity:", result["control_design_parity"])


Run: 7648bca3c639da0e050f | Decision: DO_NOT_PROMOTE_PRIMARY
Control parity: True


## 3. Policy performance and uncertainty

Both policies matter; a mean can conceal regressions. Bands cover the predeclared comparisons in this round conditional on fixed predictions, not every adaptive experiment in the project or model-refit variability.

In [4]:
display(pd.DataFrame(result["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")


,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",role_evidence,0.693097,0.254469,0.827292
7,1,No legal advice: Do not offer or request legal...,role_evidence,0.754574,0.233848,0.816172
8,0,"No Advertising: Spam, referral links, unsolici...",context_evidence,0.703470,0.247081,0.823188
9,1,No legal advice: Do not offer or request legal...,context_evidence,0.755044,0.235380,0.840367


## 4. Mechanism controls and family removals

A primary gain against raw Qwen alone is insufficient. It must also beat the answer-only readout, raw basic features and the mechanism controls. Full-minus-family comparisons quantify the separate contributions; no secondary winner replaces the registered primary.

In [5]:
display(pd.DataFrame(result["comparisons"]))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,role_evidence,frozen_basic,role_evidence vs basic geometry,0.000768,-0.014407,0.015944
1,context_evidence,frozen_basic,context_evidence vs basic geometry,0.006189,-0.008986,0.021365
2,conditioned_all,frozen_basic,conditioned_all vs basic geometry,0.005735,-0.009441,0.020910
3,permuted_all,frozen_basic,permuted_all vs basic geometry,-0.005359,-0.020534,0.009817
4,uniform_all,frozen_basic,uniform_all vs basic geometry,0.005843,-0.009333,0.021018
5,descriptor_only,frozen_basic,descriptor_only vs basic geometry,0.000519,-0.014656,0.015695
6,conditioned_all,qwen_raw,Primary vs raw Qwen,0.008909,-0.006267,0.024084
7,conditioned_all,answer_only,Primary vs answer-only,0.008909,-0.006267,0.024084
8,conditioned_all,permuted_all,Correct behavior alignment vs reference permut...,0.011093,-0.004082,0.026268
9,conditioned_all,uniform_all,Semantic affinity vs uniform reference weighting,-0.000108,-0.015284,0.015067


## 5. Reference coverage and diagnostics

Matching absence is distinct from observing a behavior. The fixed backoff avoids empty-class neighborhoods. Inspect which descriptors are rare before interpreting any coefficient.

In [6]:
display(pd.DataFrame(result["diagnostics"]).query("inner_fold == 'outer_query'"))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")


,fold,inner_fold,self_text_overlap,descriptor,references,query_rows,reference_prevalence,query_prevalence,zero_stratum_fraction,mean_minimum_coverage,backoff
36,0,outer_query,0,act_roles/self_request_legal_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
37,0,outer_query,0,act_roles/other_request_legal_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
38,0,outer_query,0,act_roles/directive_legal_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
39,0,outer_query,0,act_roles/offer_legal_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
40,0,outer_query,0,act_roles/experience_legal_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
41,0,outer_query,0,act_roles/reported_legal_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
42,0,outer_query,0,action_scope/quoted_directive_legal_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
43,0,outer_query,0,action_scope/directive_legal_negated_present,629,234,0.000000,0.000000,0.000000,1.000000,0.2
44,0,outer_query,0,action_scope/disclaimer_then_directive_legal_p...,629,234,0.000000,0.000000,0.000000,1.000000,0.2
45,0,outer_query,0,link_intent/cta_url_present,629,234,0.058824,0.072650,0.000000,0.878895,0.2


## 6. Fitted associations and probability quality

Standardized coefficients are descriptive, not causal importance. Matched additions/removals and policy stability are the evidence of feature value. Brier score and log loss may worsen even when AUC increases.

In [7]:
display(pd.DataFrame(result["pooled_metrics"]))
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,role_evidence,0.723836,0.735001,0.738218
4,context_evidence,0.729257,0.736183,0.741330
5,conditioned_all,0.728802,0.735392,0.740687
6,permuted_all,0.717709,0.702199,0.723668
7,uniform_all,0.728910,0.735914,0.740913
8,descriptor_only,0.723587,0.734801,0.737955


## 7. Fixed decision, limitations, and handoff

Primary: `conditioned_all`. Require +0.003 macro AUC, a positive simultaneous lower bound, and no per-policy regression against every registered comparator. Ranked-pooled AUC cannot decline versus raw Qwen. A pass is eligibility for further validation only.

The cached adapted **training answer margins remain in-sample** on support labels. Cross-fitting new features does not repair that limitation. These are exploratory development experiments, not independent evidence of a leaderboard gain.

See the accompanying methodology document for equations, research sources, controls, and applicability limits.

In [8]:
for requirement in result["primary_requirements"]:
    display(pd.DataFrame([requirement["contrast"]]))
    print(requirement["reference"], requirement["per_policy_delta"], requirement["passed"])
print("Decision:", result["decision"])
for limitation in result["limitations"]:
    print(limitation)
print("Dashboard:", write_dashboard(ROOT, result))
print("No automatic GPU work, Git push, or model promotion.")


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,conditioned_all,qwen_raw,Primary vs raw Qwen,0.008909,-0.006267,0.024084


qwen_raw [0.024216417910447707, -0.006399091994285788] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,conditioned_all,answer_only,Primary vs answer-only,0.008909,-0.006267,0.024084


answer_only [0.024216417910447707, -0.006399091994285788] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,conditioned_all,frozen_basic,conditioned_all vs basic geometry,0.005735,-0.009441,0.02091


...

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,conditioned_all,uniform_all,Semantic affinity vs uniform reference weighting,-0.000108,-0.015284,0.015067


uniform_all [0.00022388059701494711, -0.00044030449501963886] False


,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,conditioned_all,descriptor_only,Conditional evidence beyond direct descriptors,0.005215,-0.00996,0.020391


descriptor_only [0.010597014925373016, -0.00016633725367420915] False
Decision: DO_NOT_PROMOTE_PRIMARY
Exploratory follow-up on repeatedly inspected development data, not a Kaggle score.
Both new rounds were fixed before either result; neither selects the other round.
Cross-fitting excludes each held training group from every fitted reference statistic.
The retained adapted training answer margin is in-sample on supplied support labels.
Cross-fitting new features does not make that answer margin out-of-fold.
Inner reference pools are smaller than the outer inference pool.
Conditional intervals cover this round only, not all adaptive project choices.
A screen pass is eligibility for new validation, never automatic GPU authorization.
Behavior descriptors are approximate observable cues, never inferred moderation labels.
Behavior matching includes matching absence; rare/empty strata back off at weight 0.2.
Reference permutation preserves descriptor rows but breaks their example alignment.

Dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/behavior_support_features/dashboard.html
No automatic GPU work, Git push, or model promotion.
